# Toy saliva sharing model

With reward and cost

In [1]:
from memo import memo
import jax
import jax.numpy as jnp

import numpy as np
import pandas as pd

from enum import IntEnum

## Model

The forward planning experiment manipulates reward and the relationship between the characters.

The relationship is on a scale of 0 to 100, where 0 is completely formal and 100 is completely intimate.

In [2]:
actions = jnp.array([0, 1, 2, 3])

class RewardConditions(IntEnum):
    LOW = 0
    HIGH = 1

class RelationshipConditions(IntEnum):
    ZERO = 0
    FIFTY = 1
    SEVENTY_FIVE = 2
    ONE_HUNDRED = 3


Because the relationship is a scale of 1 to 100, but the forward planning experiment only has 4 discrete levels, let's get the intimacy levels for each of the 4 levels (and scale it between 0 and 1)

Also, get the risk for each of the 4 actions. Action 0 and 1 have no saliva transfer, so the risk is 0.

In [3]:
@jax.jit
def get_intimacy(relationship_condition):
    return jnp.array([0, 0.5, 0.75, 1])[relationship_condition]

@jax.jit
def get_risk(action):
    return jnp.array([0, 0, 1, 2])[action]

Reward and cost

- In the low reward condition, the characters don't particularly want to eat the food together. So the reward is 1 for action 0, and 0 for all other actions.
- In the high reward condition, the characters want to eat the food together. So the reward is 0 for action 0, and 1 for all other actions.

In [4]:
@jax.jit
def get_reward(action, reward_condition): 
    low_reward = jnp.array([1, 0, 0, 0])
    high_reward = jnp.array([0, 1, 1, 1])
    which_reward = jnp.where(reward_condition == RewardConditions.LOW, low_reward, high_reward)
    return which_reward[action]


@jax.jit
def get_cost(action, relationship_condition):
    """
    This is a basic discomfort term
    The more intimate the relationship, the smaller the cost.
    Most formal relationship -> keep original risk value
    Most intimate relationship -> scale down risk value
    """
    intimacy = get_intimacy(relationship_condition)
    formality = 1 - intimacy
    risk = get_risk(action)
    return formality * risk

Memo stuff

In [5]:
@memo
def vanilla_actor[
    action: actions,
    relationship_condition: RelationshipConditions,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor]
    actor: knows(relationship_condition)
    actor: knows(reward_condition)
    actor: chooses(
        action in actions,
        wpp=exp(
            alpha
            * (w_r * get_reward(action, reward_condition) - w_c * get_risk(action))
        ),
    )

    return Pr[actor.action == action]

@memo
def relationship_actor[
    action: actions,
    relationship_condition: RelationshipConditions,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor]
    actor: knows(relationship_condition)
    actor: knows(reward_condition)
    actor: chooses(
        action in actions,
        wpp=exp(
            alpha
            * (
                w_r * get_reward(action, reward_condition)
                - w_c * get_cost(action, relationship_condition)
            )
        ),
    )

    return Pr[actor.action == action]

## Generate predictions

### Actor predictions

In [6]:
params = {
    "alpha": 1,
    "w_r": 1,
    "w_c": 1,
}

output_mappings = {
    "ZERO": 0,
    "FIFTY": 50,
    "SEVENTY_FIVE": 75,
    "ONE_HUNDRED": 100,
    "LOW": "low",
    "HIGH": "high"
}

In [7]:
vanilla_result = vanilla_actor(
    alpha=params["alpha"], w_r=params["w_r"], w_c=params["w_c"], return_pandas=True
)
relationship_result = relationship_actor(
    alpha=params["alpha"], w_r=params["w_r"], w_c=params["w_c"], return_pandas=True
)

df_vanilla = (
    vanilla_result[1]
    .pandas.rename(
        columns={
            "relationship_condition": "intimacy",
            "reward_condition": "reward",
            "la_actor": "p_action",
        }
    )
    .replace(output_mappings)
)
df_vanilla["model"] = "vanilla"

df_relationship = (
    relationship_result[1]
    .pandas.rename(
        columns={"relationship_condition": "intimacy", "reward_condition": "reward", "ionship_actor": "p_action"}
    )
    .replace(output_mappings)
)
df_relationship["model"] = "relationship"

df_combined = pd.concat([df_vanilla, df_relationship])

# add alpha, w_r, w_c to the beginning of the dataframe
df_combined.insert(0, "w_c", params["w_c"])
df_combined.insert(0, "w_r", params["w_r"])
df_combined.insert(0, "alpha", params["alpha"])

In [8]:
df_combined

,alpha,w_r,w_c,action,intimacy,reward,p_action,model
0,1,1,1,0,0,low,0.643914,vanilla
1,1,1,1,0,0,high,0.196612,vanilla
2,1,1,1,0,50,low,0.643914,vanilla
3,1,1,1,0,50,high,0.196612,vanilla
4,1,1,1,0,75,low,0.643914,vanilla
...,...,...,...,...,...,...,...,...
27,1,1,1,3,50,high,0.157060,relationship
28,1,1,1,3,75,low,0.118843,relationship
29,1,1,1,3,75,high,0.220299,relationship
30,1,1,1,3,100,low,0.174878,relationship


In [9]:
df_combined.to_csv("toy_model_predictions.csv", index=False)